# 📄 ID-VLM — Notebook 01: Data Preparation

**Goal:** Download the MIDV-2020 dataset, explore the annotation structure, and convert raw images + annotations into Unsloth-compatible instruction/answer pairs.

**Output:** `train.jsonl`, `val.jsonl`, `test.jsonl` saved to Google Drive.

---

## What this notebook does:
1. Mount Google Drive & clone the project repo
2. Download MIDV-2020 dataset (rectified photos)
3. Explore annotations and image quality
4. Convert to ChatML instruction pairs
5. Split into train/val/test and save

## 1. Setup & Mount Drive

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Project directory on Drive
import os
PROJECT_DIR = '/content/drive/MyDrive/id-vlm'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data/raw', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data/processed', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/outputs', exist_ok=True)
print(f'Project directory: {PROJECT_DIR}')

In [ ]:
# Clone the project repo
REPO_DIR = '/content/id-vlm'
if not os.path.exists(f'{REPO_DIR}/config.py'):
    print('Cloning ID-VLM repository...')
    !git clone https://github.com/OmTilwar/ID-VLM.git {REPO_DIR}
else:
    print('✅ Repo already cloned at', REPO_DIR)

In [ ]:
# Install dependencies
!pip install Pillow python-Levenshtein numpy tqdm -q

import sys
sys.path.insert(0, REPO_DIR)

## 2. Download MIDV-2020 Dataset

We download the official MIDV-2020 Rectified Photos dataset archive (~560 MB) directly into our workspace.

In [ ]:
# ── Download MIDV-2020 Rectified Photos Dataset ──
RAW_DIR = f'{PROJECT_DIR}/data/raw'
TAR_URL = 'https://www.irisa.fr/intuidoc/data/database/rectified_photos.tar.xz'
DOC_TYPES = ['alb_id', 'aze_passport', 'esp_id', 'grc_passport']

print('Downloading MIDV-2020 Rectified Photos dataset (~560 MB)...')
os.makedirs(RAW_DIR, exist_ok=True)

target_archive = f'{RAW_DIR}/rectified_photos.tar.xz'
extracted_dir = f'{RAW_DIR}/rectified_photos'

if not os.path.exists(extracted_dir):
    if not os.path.exists(target_archive):
        print(f'Downloading from {TAR_URL}...')
        !wget -O {target_archive} {TAR_URL} || curl -o {target_archive} {TAR_URL}
    
    if os.path.exists(target_archive):
        print('Extracting dataset archive...')
        !tar -xf {target_archive} -C {RAW_DIR}
    else:
        print('⚠️ Download failed — pipeline will automatically generate synthetic ID dataset fallback.')
else:
    print('✅ Dataset already downloaded and extracted!')

print(f'\nRaw data directory contents:')
!ls -l {RAW_DIR}

## 3. Explore the Dataset

Let me inspect the downloaded annotations and image files.

In [ ]:
import glob
import json
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

print(f'Searching for images and annotations in {RAW_DIR}...')

image_files = glob.glob(os.path.join(RAW_DIR, '**', '*.jpg'), recursive=True) \
            + glob.glob(os.path.join(RAW_DIR, '**', '*.png'), recursive=True)

json_files = glob.glob(os.path.join(RAW_DIR, '**', '*.json'), recursive=True)

print(f'Found {len(image_files)} images, {len(json_files)} JSON files')

if json_files:
    print(f'\nSample JSON: {json_files[0]}')
    with open(json_files[0], 'r') as f:
        sample_annotation = json.load(f)
    # Print snippet
    snippet = json.dumps(sample_annotation, indent=2)
    print(snippet[:1500])

In [ ]:
# Visualize sample images
if image_files:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    for i, img_path in enumerate(image_files[:8]):
        img = Image.open(img_path)
        axes[i].imshow(img)
        axes[i].set_title(f'{Path(img_path).name}\n{img.size}', fontsize=8)
        axes[i].axis('off')
        
    for j in range(len(image_files[:8]), len(axes)):
        axes[j].axis('off')
        
    plt.suptitle('MIDV-2020 Sample Identity Documents', fontsize=14)
    plt.tight_layout()
    plt.show()

## 4. Convert to ChatML Instruction Pairs

In [ ]:
from src.dataset import create_dataset, split_dataset, save_dataset
import config

# Override paths
config.RAW_DATA_DIR = RAW_DIR
config.PROCESSED_DATA_DIR = f'{PROJECT_DIR}/data/processed'

# Create dataset
print('Converting raw dataset to ChatML instruction pairs...')
dataset = create_dataset(data_dir=RAW_DIR, doc_types=DOC_TYPES)

print(f'\nTotal instruction pairs: {len(dataset)}')

# Inspect sample pair
if dataset:
    sample = dataset[0]
    print('\nSample Instruction Pair:')
    print('User Prompt:', sample['messages'][0]['content'][1]['text'])
    print('Expected JSON:', sample['messages'][1]['content'][0]['text'])

## 5. Split & Save

In [ ]:
# Split into train/val/test
train, val, test = split_dataset(dataset, 0.7, 0.15, 0.15)

print(f'Train set: {len(train)} samples')
print(f'Val set:   {len(val)} samples')
print(f'Test set:  {len(test)} samples')

# Save to Drive
output_dir = f'{PROJECT_DIR}/data/processed'
os.makedirs(output_dir, exist_ok=True)

save_dataset(train, f'{output_dir}/train.jsonl')
save_dataset(val, f'{output_dir}/val.jsonl')
save_dataset(test, f'{output_dir}/test.jsonl')
save_dataset(dataset, f'{output_dir}/full.jsonl')

print(f'\n✅ Saved JSONL files to {output_dir}/')
!ls -lh {output_dir}/